In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# Loading balanced dataset — we will use a 20k sample for topic modeling
# Larger samples give better topics but slower training
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/yelp_NLP/reviews_balanced_50k.csv')

# Sample 20k for topic modeling — good balance of speed vs quality
sample = df.sample(20000, random_state=42).reset_index(drop=True)
print(f"Working with {len(sample):,} reviews")
print(sample['sentiment'].value_counts())

Working with 20,000 reviews
sentiment
positive    8054
neutral     6006
negative    5940
Name: count, dtype: int64


In [ ]:
# BERTopic pipeline: embeddings -> UMAP dimensionality reduction ->
# HDBSCAN clustering -> TF-IDF topic words
# This cell takes 10-15 minutes on Colab GPU
!pip install bertopic sentence-transformers umap-learn hdbscan plotly -q
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

# Embedding model — converts each review to a 384-dim semantic vector
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# UMAP reduces 384 dimensions down to 5 for clustering
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# HDBSCAN finds dense clusters — reviews about similar things end up in same cluster
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric='euclidean',
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    nr_topics='auto',       # let the model decide how many topics
    verbose=True
)

topics, probs = topic_model.fit_transform(sample['clean_text'].tolist())
print(f"\nNumber of topics found: {len(set(topics)) - 1}")  # -1 excludes outlier topic

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-06-09 09:42:59,174 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

2026-06-09 10:05:54,796 - BERTopic - Embedding - Completed ✓
2026-06-09 10:05:54,804 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-09 10:06:50,068 - BERTopic - Dimensionality - Completed ✓
2026-06-09 10:06:50,070 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-09 10:06:59,735 - BERTopic - Cluster - Completed ✓
2026-06-09 10:06:59,737 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-06-09 10:07:01,346 - BERTopic - Representation - Completed ✓
2026-06-09 10:07:01,348 - BERTopic - Topic reduction - Reducing number of topics
2026-06-09 10:07:01,373 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-09 10:07:02,875 - BERTopic - Representation - Completed ✓
2026-06-09 10:07:02,880 - BERTopic - Topic reduction - Reduced number of topics from 60 to 7



Number of topics found: 6


In [ ]:
# Viewing the top words per topic — this tells us what each cluster is actually about
topic_info = topic_model.get_topic_info()
print(f"Total topics: {len(topic_info)}")
print("\nTop 15 topics by size:")
print(topic_info.head(16).to_string())  # row 0 is outliers, skip it

Total topics: 7

Top 15 topics by size:
   Topic  Count                    Name                                            Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [ ]:
# Drilling into each topic's keywords to understand what the cluster represents
# Topic -1 is always the outlier bucket — skip it
print("TOP WORDS PER TOPIC")
print("=" * 60)

for topic_id in topic_info['Topic'].values[1:16]:  # top 15 topics
    words = topic_model.get_topic(topic_id)
    word_list = [w[0] for w in words[:8]]
    count = topic_info[topic_info['Topic'] == topic_id]['Count'].values[0]
    print(f"Topic {topic_id:>3} ({count:>5} reviews): {', '.join(word_list)}")

TOP WORDS PER TOPIC
Topic   0 (11052 reviews): the, and, to, was, of, it, for, is
Topic   1 (  556 reviews): and, my, to, the, hair, she, was, for
Topic   2 (  112 reviews): and, gym, to, the, classes, is, you, of
Topic   3 (   87 reviews): dog, to, and, the, they, my, for, of
Topic   4 (   79 reviews): car, wash, the, and, to, my, they, that
Topic   5 (   57 reviews): closed, open, they, the, at, to, and, door


In [ ]:
topic_labels = {
    0: "General Dining",
    1: "Beauty and Salons",
    2: "Gym and Fitness",
    3: "Pet Services",
    4: "Car Wash",
    5: "Business Hours Issues",
}

In [ ]:
# Adding topic assignments back to the dataframe for analysis
sample['topic_id'] = topics
sample['topic_label'] = sample['topic_id'].map(topic_labels).fillna('Other')

print("Topic distribution in sample:")
print(sample['topic_label'].value_counts().head(15))

Topic distribution in sample:
topic_label
General Dining           11052
Other                     8057
Beauty and Salons          556
Gym and Fitness            112
Pet Services                87
Car Wash                    79
Business Hours Issues       57
Name: count, dtype: int64
